In [0]:
# CSV - Row Based File Format

In [0]:
# Parquet - Columnar Based File Format and also comes with compression techniques (default snappy) which stores data in less space

In [0]:
# Delta File Format - ACID Transaction, Schema Drift and Time Travel | internally these delta files/tables uses parquet file format to store data

In [0]:
# ACID Transactions

1. Atomicity - All or nothing
2. Consistency - Data is always consistent
3. Isolation - Transactions are isolated from each other
4. Durability - Once a transaction is committed


# Schema Drift

1. Schema Drift is a concept where the schema of the data changes over time.
2. Schema Drift can be handled by Delta Lake using Schema Evolution.
3. Schema Evolution is a feature of Delta Lake that allows you to evolve the schema of your data over time without breaking existing queries.
4. Schema Evolution is enabled by default in Delta Lake.

# Time Travel

1. Time Travel is a feature of Delta Lake that allows you to query data at a specific point in time.
2. Time Travel is enabled by default in Delta Lake.
3. Time Travel is useful for debugging and auditing purposes.

In [0]:
# delete from patient

In [0]:
dbutils.fs.mounts()

In [0]:
%fs
ls "/mnt/rainbow-container"

In [0]:
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/mnt/rainbow-container/bihar_election_results.csv")

schema = [col.replace(" ", "") for col in df.columns]
df = df.toDF(*schema)

df.display()

In [0]:
df.write.format("delta").mode("overwrite").save("/mnt/rainbow-container/delta_files/bihar_elections")

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("hive_metastore.rainbow.bihar_elections")

In [0]:
df = df.filter(F.col("ConstituencyNumber") == 1)

In [0]:
from pyspark.sql import functions as F

In [0]:
# df = (
#     df.groupBy("ConstituencyNumber", "ConstituencyName")
#     .agg(F.sum("TotalVotes").alias("TotalVotes"))
# )

# df.display()

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("hive_metastore.rainbow.bihar_elections")

In [0]:
%sql

select * from hive_metastore.rainbow.bihar_elections;

In [0]:
%sql
desc history hive_metastore.rainbow.bihar_elections;

In [0]:
%sql
-- Time Travel

select * from hive_metastore.rainbow.bihar_elections VERSION AS OF 0;

In [0]:
%sql
restore table hive_metastore.rainbow.bihar_elections to version as of 0;

In [0]:
%sql
select * from hive_metastore.rainbow.bihar_elections;

In [0]:
adls_df = spark.read.format("delta").load("/mnt/rainbow-container/delta_files/bihar_elections")
display(adls_df)

In [0]:
%sql
DESCRIBE HISTORY delta.`/mnt/rainbow-container/delta_files/bihar_elections`;

In [0]:
%sql
restore table delta.`/mnt/rainbow-container/delta_files/bihar_elections` version as of 0;

In [0]:
adls_df = spark.read.format("delta").load("/mnt/rainbow-container/delta_files/bihar_elections")
display(adls_df)

In [0]:
1. Optimize - merge small files into large files
2. Vaccum - delete files which are unused
3. Z-Order - physically it clusters data for efficient filitering 

In [0]:
from pyspark.sql import functions as F


data = (
    spark.range(1, 1000000)
    .withColumn("customer_id", (F.rand()*1000).cast("int"))
    .withColumn("city", F.expr("case when customer_id < 300 then 'Bangalore' \
        when customer_id < 600 then 'Mumbai' \
        when customer_id < 800 then 'Delhi' \
        else 'Chennai' end"))
    .withColumn("amount", (F.rand()*100000).cast("int"))
    .drop("id")
)

data.write.mode("overwrite").save("/mnt/rainbow-container/delta/customer_data")

In [0]:
# %sql
# alter table hive_metastore.rainbow.customer_data
# drop column id;

In [0]:
for i in range(20):
    data.sample(0.05).write.format("delta").mode("append").save("/mnt/rainbow-container/delta/customer_data")

In [0]:
%sql
optimize hive_metastore.rainbow.customer_data;

In [0]:
%sql
optimize delta.`/mnt/rainbow-container/delta/customer_data`;

In [0]:
%sql
vacuum delta.`/mnt/rainbow-container/delta/customer_data` retain 0 hours;

In [0]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

In [0]:
%sql
vacuum delta.`/mnt/rainbow-container/delta/customer_data` retain 0 hours;

In [0]:
%sql

select * from delta.`/mnt/rainbow-container/delta/customer_data`
where customer_id = 254;

In [0]:
%sql
optimize delta.`/mnt/rainbow-container/delta/customer_data`
zorder by (customer_id);

In [0]:
%sql
select * from delta.`/mnt/rainbow-container/delta/customer_data`
where customer_id = 254;